# Big Data with Dask

**What you'll learn:** How to use Dask to analyze large datasets with a pandas-like API, understand lazy evaluation, and see how Dask parallelizes work.

**Prerequisites:** [01 - Python Basics](01-python-basics.ipynb), [02 - Big Data Intro](02-big-data-intro.ipynb)

**Before you start:** Make sure you've generated the large dataset:
```bash
python scripts/generate_large_data.py
```

## What is Dask?

Dask is a Python library for **parallel computing**. Its DataFrame module is designed to feel like pandas, but it splits your data into **partitions** (chunks) and processes them across all your CPU cores simultaneously.

If you already know pandas, Dask is the easiest path to scaling up.

## Reading Data

The syntax is almost identical to pandas:

In [ ]:
import dask.dataframe as dd

ddf = dd.read_csv("../data/sales_large.csv")

print(type(ddf))
print(f"Columns: {list(ddf.columns)}")
print(f"Number of partitions: {ddf.npartitions}")

Notice that Dask reports **partitions** instead of a row count. Each partition is a chunk of the data that can be processed independently.

## Lazy Evaluation: The Key Concept

This is the most important thing to understand about Dask. When you write operations on a Dask DataFrame, **nothing actually happens yet**. Dask just records what you want to do. The computation only runs when you call `.compute()`.

Think of it like writing a recipe: you list all the steps first, then execute them all at once.

In [ ]:
# This does NOT compute anything yet — it's just a plan
electronics = ddf[ddf["category"] == "Electronics"]
print(type(electronics))  # still a Dask DataFrame, not a result

In [ ]:
# .compute() triggers the actual work and returns a pandas DataFrame
electronics_pandas = electronics.compute()
print(type(electronics_pandas))   # now it's a regular pandas DataFrame
print(f"Electronics rows: {len(electronics_pandas):,}")

`.head()` is an exception — it always computes immediately (it only needs to read the first partition):

In [ ]:
ddf.head(5)

## Familiar pandas Operations

Most pandas operations work the same way in Dask. The difference is that you need `.compute()` at the end to get the result.

### Filtering

In [ ]:
# Big orders: quantity > 7 and price > 50
big_orders = ddf[(ddf["quantity"] > 7) & (ddf["unit_price"] > 50)]
result = big_orders.compute()
print(f"Big orders: {len(result):,}")
result.head()

### Groupby and Aggregation

In [ ]:
# Average price and total quantity by category
summary = ddf.groupby("category").agg(
    {"unit_price": "mean", "quantity": "sum"}
).compute()

summary.columns = ["avg_price", "total_quantity"]
summary = summary.round(2)
print(summary)

### Adding New Columns

In [ ]:
ddf["total_price"] = ddf["quantity"] * ddf["unit_price"]
ddf[["product", "quantity", "unit_price", "total_price"]].head(5)

### Value Counts

In [ ]:
region_counts = ddf["region"].value_counts().compute()
print(region_counts)

## Dask vs pandas Side-by-Side

Here's the same operation in both, so you can see how similar they are:

In [ ]:
import pandas as pd

# --- pandas ---
pdf = pd.read_csv("../data/sales_large.csv")
pandas_result = pdf.groupby("category")["unit_price"].mean()
print("pandas result:")
print(pandas_result.round(2))

print()

# --- Dask ---
dask_result = ddf.groupby("category")["unit_price"].mean().compute()
print("Dask result:")
print(dask_result.round(2))

Same result, same syntax, just add `.compute()` at the end.

## Timing Comparison

In [ ]:
import time

# pandas
start = time.time()
pdf.groupby(["category", "region"])["unit_price"].agg(["mean", "sum", "count"])
pandas_time = time.time() - start

# Dask
start = time.time()
ddf.groupby(["category", "region"])["unit_price"].agg(["mean", "sum", "count"]).compute()
dask_time = time.time() - start

print(f"pandas: {pandas_time:.3f}s")
print(f"Dask:   {dask_time:.3f}s")
print()
print("At 100K rows, pandas is often faster due to Dask's coordination overhead.")
print("Dask's advantage appears at millions of rows, especially on multi-core machines.")

## Understanding Partitions

Dask splits your data into **partitions**. Each partition is a regular pandas DataFrame. Dask processes partitions in parallel across your CPU cores.

In [ ]:
print(f"Number of partitions: {ddf.npartitions}")

# You can look at a single partition
first_partition = ddf.get_partition(0).compute()
print(f"Rows in first partition: {len(first_partition)}")
print(f"Type: {type(first_partition)}")

You can control the number of partitions when reading data:

```python
ddf = dd.read_csv("file.csv", blocksize="10MB")  # roughly 10 MB per partition
```

More partitions = more parallelism, but also more coordination overhead. The default usually works well.

## When to Use Dask

**Choose Dask when:**
- You already know pandas and want to scale up with minimal code changes
- Your data is too large for pandas but fits on one machine (GBs, not TBs)
- You want parallel execution without setting up Spark or a cluster

**Choose something else when:**
- Data is small enough for pandas (just use pandas)
- Data requires a distributed cluster (use PySpark)
- You mainly need SQL queries on files (use DuckDB)

---

## Key Takeaways

- Dask looks and feels like pandas, but works in parallel.
- Operations are **lazy** — nothing runs until you call `.compute()`.
- Data is split into **partitions** processed across CPU cores.
- At 100K rows, overhead may make it slower than pandas. The payoff comes with larger data.

---

**Next up:** [05 - Big Data with DuckDB](05-big-data-duckdb.ipynb) — fast SQL analytics with zero setup.